# v5 双 ETF 全分钟来源与新信息边界

冻结 `plan.json` 后只读 Qlib provider 的 2018–2020 发现期。核查可用原始频道、原价与复权价之间的关系和真实决策时钟；2021/2022 留存，零标签、拟合或账户。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
root=Path.cwd();card=root/"research/stage1_v5_minute_source_novelty_20261030";plan=json.loads((card/"plan.json").read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
provider=root/plan["provider_uri"];source=root/plan["source_card"]
assert plan["frozen"] and sha(provider/"qlib_store_manifest.json")==plan["provider_manifest_sha256"]
assert sha(source/"summary.json")==plan["source_summary_sha256"]
assert json.loads((source/"summary.json").read_text())["nas_evidence"]["sha256"]==plan["source_nas_archive_sha256"]
files={d.name:{f.name.removesuffix(".1min.bin") for f in d.glob("*.1min.bin")} for d in (provider/"features").iterdir() if d.is_dir()}
assert set(files)==set(x.lower() for x in plan["instruments"])
assert all(files[s]==set(plan["fields"]) for s in files)
qlib.init(provider_uri=str(provider),region="us",kernels=1)
fields=["$"+x for x in plan["fields"]]
bars=D.features(plan["instruments"],fields,start_time=plan["discovery_start"]+" 09:30:00",end_time=plan["discovery_end"]+" 16:00:00",freq="1min")
assert len(bars)>0 and set(bars.index.get_level_values("instrument"))==set(plan["instruments"])
print("Qlib field inventory",files,"rows",len(bars))

[19463:MainThread](2026-10-30 13:12:05,458) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[19463:MainThread](2026-10-30 13:12:05,887) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[19463:MainThread](2026-10-30 13:12:05,888) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


Qlib field inventory {'soxx': {'close', 'open', 'raw_volume', 'volume', 'low', 'raw_low', 'raw_open', 'high', 'price_valid', 'raw_high', 'raw_close'}, 'soxl': {'close', 'open', 'raw_volume', 'volume', 'low', 'raw_low', 'raw_open', 'high', 'price_valid', 'raw_high', 'raw_close'}} rows 432720


In [2]:
rows=[]
for sym in plan["instruments"]:
    b=bars.xs(sym,level="instrument").copy()
    b.index=pd.to_datetime(b.index)
    valid=b["$price_valid"].eq(1)
    complete=np.ones(len(b),bool)
    for fld in plan["fields"]:
        complete &= np.isfinite(b["$"+fld].to_numpy(float))
    for day,g in b.groupby(b.index.date,sort=True):
        i=b.index.date==day
        v=valid.to_numpy()[i]&complete[i]
        if not v.any():continue
        x=g.iloc[np.flatnonzero(v)]
        raw=x[["$raw_"+c for c in ("open","high","low","close")]].to_numpy(float)
        adjusted=x[["$"+c for c in ("open","high","low","close")]].to_numpy(float)
        ratio=raw/adjusted
        median=np.median(ratio)
        spread=float(np.max(np.abs(ratio/median-1)))
        rawvol=x["$raw_volume"].to_numpy(float);vol=x["$volume"].to_numpy(float)
        volmask=(rawvol>0)&(vol>0)
        volume_ratio=np.nan if not volmask.any() else float(np.median(rawvol[volmask]/vol[volmask]))
        volume_ratio_spread=np.nan if not volmask.any() else float(np.max(np.abs((rawvol[volmask]/vol[volmask])/volume_ratio-1)))
        rows.append({"symbol":sym,"date":str(day),"valid_bars":int(len(x)),"raw_adjusted_ohlc_median_ratio":float(median),"raw_adjusted_ohlc_max_relative_spread":spread,"raw_adjusted_volume_median_ratio":volume_ratio,"raw_adjusted_volume_max_relative_spread":volume_ratio_spread})
result=pd.DataFrame(rows)
paired=result.groupby("date").symbol.nunique().eq(2).sum()
summary={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),"qlib_minute_rows":int(len(bars)),"pair_days":int(paired),"symbols":{},"raw_channels_per_symbol":5,"independent_market_instruments":2,"historical_first_receipt_qualified":False,"discovery_years":[2018,2019,2020],"reserved_years_untouched":plan["reserved_years"],"new_model_fits":0,"new_native_accounts":0,"new_recorders":0}
for sym in plan["instruments"]:
    z=result[result.symbol==sym]
    summary["symbols"][sym]={"days":int(len(z)),"valid_bars":int(z.valid_bars.sum()),"maximum_daily_ohlc_ratio_spread":float(z.raw_adjusted_ohlc_max_relative_spread.max()),"maximum_daily_volume_ratio_spread":float(z.raw_adjusted_volume_max_relative_spread.max())}
assert paired>=plan["gate"]["minimum_valid_pair_days"]
assert all(v["maximum_daily_ohlc_ratio_spread"]<=plan["gate"]["maximum_relative_raw_adjusted_ohlc_ratio_spread_per_day"] for v in summary["symbols"].values())
summary["decision"]="raw_and_adjusted_prices_are_same_intraday_information_channel_with_source_receipt_unknown"
(card/"audit.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n")
print(json.dumps(summary,ensure_ascii=False,indent=2))

{
  "experiment": "stage1_v5_minute_source_novelty_20261030",
  "plan_sha256": "3d33399bee3355a63e2884c7291411ddfb5cd6407b1b5315e460f882c2f0a734",
  "qlib_minute_rows": 432720,
  "pair_days": 558,
  "symbols": {
    "SOXL": {
      "days": 558,
      "valid_bars": 216360,
      "maximum_daily_ohlc_ratio_spread": 1.1766244123467828e-07,
      "maximum_daily_volume_ratio_spread": 0.0
    },
    "SOXX": {
      "days": 558,
      "valid_bars": 216359,
      "maximum_daily_ohlc_ratio_spread": 1.0436797182045154e-07,
      "maximum_daily_volume_ratio_spread": 0.0
    }
  },
  "raw_channels_per_symbol": 5,
  "independent_market_instruments": 2,
  "historical_first_receipt_qualified": false,
  "discovery_years": [
    2018,
    2019,
    2020
  ],
  "reserved_years_untouched": [
    2021,
    2022
  ],
  "new_model_fits": 0,
  "new_native_accounts": 0,
  "new_recorders": 0,
  "decision": "raw_and_adjusted_prices_are_same_intraday_information_channel_with_source_receipt_unknown"
}


## 中文结论

SOXX 判定：本卡无新模型或候选账户，合同 v2 的 0/2/5bp 门均未检验。Qlib 在 2018–2020 发现期读取双 ETF 432720 行，558 个共同有效日；SOXL 有效分钟 216360，SOXX 216359，差的一根是已知无效 bar。每只 ETF 的原价/复权 OHLC 日内比例最大相对漂移分别 1.18e−7/1.04e−7，成交量比例日内漂移为 0。原价列是价格坐标变换，不是额外的独立盘中来源。

在此 provider 内，独立交易品种只有 SOXL 与 SOXX，市场字段只有 OHLCV；`price_valid` 是来源质量掩码。既有路线已对普通分钟涨跌/量、SOXX VWAP 价格时序、下行占比/路径能量、跟踪偏离、成交量时序和分钟网络接收分别做过固定检验，见 `route_overlap.json`。它们的失败只关闭对应公式、标签与账户，不能证明任何 OHLCV 非线性函数都无效。

可用时钟仍是假设：完成分钟右边界加 60 秒送达、再下一 bar 开盘成交，历史真实首收与报价/成交回执未认证。2021/2022 未读取。本轮无法仅靠“raw 字段”或换 OHLCV 窗口提出独立信息机制；下一步优先核查逐层点时账户**连续高水位回撤余量**是否超出现有类别风险状态/持仓控制，并检索已训练的风险余量路线。若只是旧风险状态重编码，则标 `needs_distinct_decision_time_mechanism`，不拟合。